In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

from src.utils.config import Config

print("✅ Ready")

## 1. Load and Prepare Data

In [ ]:
config = Config()
df = pd.read_parquet(config.data_dir / 'staging' / 'test_results.parquet')

print(f"Dataset: {len(df):,} records")
print(f"Period: {df.index.min()} to {df.index.max()}" if isinstance(df.index, pd.DatetimeIndex) else "")
df.head()

## 2. Univariate Analysis

Analyze each variable individually.

In [ ]:
# Distribution of test results
result_counts = df['result'].value_counts()
result_pct = (result_counts / len(df) * 100).round(1)

print("Test Result Distribution:")
for result, count in result_counts.items():
    print(f"  {result}: {count:,} ({result_pct[result]}%)")

fig = px.pie(
    values=result_counts.values,
    names=result_counts.index,
    title='Test Result Distribution',
    color=result_counts.index,
    color_discrete_map={'pass': 'green', 'fail': 'red'}
)
fig.show()

In [ ]:
# Test time distribution
print("\nTest Time Statistics:")
print(df['test_time_ms'].describe())

fig = px.histogram(
    df,
    x='test_time_ms',
    nbins=50,
    title='Test Time Distribution',
    marginal='box',
    labels={'test_time_ms': 'Test Time (ms)'}
)
fig.show()

In [ ]:
# Bin distribution
bin_counts = df['bin'].value_counts().sort_index()

fig = px.bar(
    x=bin_counts.index,
    y=bin_counts.values,
    title='Bin Distribution',
    labels={'x': 'Bin', 'y': 'Count'},
    text=bin_counts.values
)
fig.update_traces(textposition='outside')
fig.show()

## 3. Bivariate Analysis

Explore relationships between variables.

In [ ]:
# Test time by result
fig = px.box(
    df,
    x='result',
    y='test_time_ms',
    title='Test Time by Result',
    color='result',
    color_discrete_map={'pass': 'green', 'fail': 'red'}
)
fig.show()

# Statistical test
pass_times = df[df['result'] == 'pass']['test_time_ms']
fail_times = df[df['result'] == 'fail']['test_time_ms']
t_stat, p_value = stats.ttest_ind(pass_times, fail_times)

print(f"\nT-test: Pass vs Fail test times")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")
print(f"  Significant? {'Yes' if p_value < 0.05 else 'No'}")

In [ ]:
# Yield by test
test_yield = df.groupby('test_name').agg({
    'result': lambda x: (x == 'pass').sum() / len(x) * 100,
    'device_id': 'count'
}).rename(columns={'result': 'yield', 'device_id': 'count'})

test_yield = test_yield.sort_values('yield')

fig = px.bar(
    test_yield,
    x=test_yield.index,
    y='yield',
    title='Yield by Test',
    labels={'yield': 'Yield (%)', 'index': 'Test Name'},
    text='yield',
    hover_data=['count']
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.add_hline(y=90, line_dash="dash", line_color="red", annotation_text="Target: 90%")
fig.show()

## 4. Multivariate Analysis

In [ ]:
# Device-level analysis
device_summary = df.groupby('device_id').agg({
    'result': lambda x: (x == 'pass').sum() / len(x) * 100,
    'test_time_ms': 'sum',
    'bin': 'first'
}).rename(columns={'result': 'yield', 'test_time_ms': 'total_time'})

fig = px.scatter(
    device_summary,
    x='total_time',
    y='yield',
    color='bin',
    title='Device Yield vs Total Test Time',
    labels={'total_time': 'Total Test Time (ms)', 'yield': 'Device Yield (%)'},
    hover_data=['bin']
)
fig.show()

In [ ]:
# Correlation analysis for numeric features
numeric_cols = ['test_time_ms', 'bin', 'test_num']
if 'measured_value' in df.columns:
    numeric_cols.append('measured_value')

corr_matrix = df[numeric_cols].corr()

fig = px.imshow(
    corr_matrix,
    title='Correlation Matrix',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto='.2f'
)
fig.show()

## 5. Key Insights & Findings

In [ ]:
# Calculate key metrics
overall_yield = (df['result'] == 'pass').sum() / len(df) * 100
device_yield = (device_summary['yield'] == 100).sum() / len(device_summary) * 100
avg_test_time = df['test_time_ms'].mean()
total_test_time = device_summary['total_time'].mean()

print("="*60)
print("KEY INSIGHTS")
print("="*60)
print(f"\n📊 Overall Metrics:")
print(f"   Test-level yield: {overall_yield:.2f}%")
print(f"   Device-level yield: {device_yield:.2f}%")
print(f"   Avg test time: {avg_test_time:.1f} ms")
print(f"   Avg total device test time: {total_test_time:.1f} ms")

print(f"\n🎯 Top Performing Tests:")
top_tests = test_yield.nlargest(3, 'yield')
for test, row in top_tests.iterrows():
    print(f"   {test}: {row['yield']:.1f}%")

print(f"\n⚠️  Tests Needing Attention:")
low_tests = test_yield.nsmallest(3, 'yield')
for test, row in low_tests.iterrows():
    print(f"   {test}: {row['yield']:.1f}%")

print(f"\n🔍 Bin Analysis:")
bin_1_pct = (df['bin'] == 1).sum() / len(df) * 100
print(f"   Bin 1 (pass): {bin_1_pct:.1f}%")
print(f"   Other bins: {100 - bin_1_pct:.1f}%")

## 6. Summary

**What we learned:**
- ✅ Univariate analysis (single variable distributions)
- ✅ Bivariate analysis (relationships between 2 variables)
- ✅ Multivariate analysis (multiple variables)
- ✅ Statistical significance testing
- ✅ Correlation analysis
- ✅ Insight generation from data

**Key Findings:**
- Overall test yield: **{:.2f}%**
- Device yield: **{:.2f}%**
- Identified tests requiring optimization
- Test time patterns analyzed

**Next Steps:**
- Notebook 04: Yield Analytics (Deep Dive)
- Build automated reporting